In [ ]:
%pip check

In [ ]:
# Get Data from Lakehouse Silver Layer to visualize and analyse features of dataset
df = spark.read.table("lh_CustomerChurn.silver.customerchurn").toPandas()
display(df.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check the class balance
sns.countplot(df, x='cr8b0_churnstatus')


In [ ]:
sns.violinplot(df, x='cr8b0_churnstatus', y='cr8b0_tenuremonths')


In [ ]:
df = df.drop(['cr8b0_customerid','cr8b0_customersubscriptionid'], axis=1)
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')

In [ ]:
df["churnstatus"] = df["cr8b0_churnstatus"].astype(str)
sns.countplot(df, x='cr8b0_contracttype', hue='churnstatus')

In [ ]:
import mlflow.sklearn
import pandas as pd
from decimal import Decimal
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.feature_selection import SelectFromModel

# 1. Get data as Panda DataFrame from the Silver Layer Lakehouse table
df = spark.read.table("lh_CustomerChurn.silver.customerchurn").toPandas()

# 1b. Convert decimal-columns (coming as object/decimal from Spark) into float
decimal_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, Decimal)).any()]
print(f"Decimal-columns converted: {decimal_cols}")
df[decimal_cols] = df[decimal_cols].astype(float)

# 2. Nulls behandeln (mit Logging)
rows_before = len(df)
df = df.dropna()
print(f"Deleted rows (Nulls): {rows_before - len(df)} von {rows_before}")

# 3. Define X / y 
X = df.drop(columns=['cr8b0_churnstatus', 'cr8b0_customerid', 'cr8b0_customersubscriptionid'])
y = df['cr8b0_churnstatus']

# 4. Train/Test-Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Preprocessing Detect categorical columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Kategoriale Spalten: {categorical_features}")

categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
preprocessor = ColumnTransformer(
    transformers=[("cat", categorical_transformer, categorical_features)],
    remainder="passthrough",
)

mlflow.set_experiment("ML_CustomerChurn-1821")

# 6. Model
# ============================================================
# MODELL 1: RandomForest with Hyperparameter-Tuning & Feature Selection
# ============================================================
pipeline_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("feature_selection", SelectFromModel(
        XGBClassifier(random_state=42, eval_metric="logloss"),
        threshold="median"  # behält nur Features über dem Median der Importance
    )),
    ("classifier", RandomForestClassifier(random_state=42, class_weight="balanced")),
])

param_dist_rf = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__max_depth": [5, 10, 15, 20, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2"],
}

search_rf = RandomizedSearchCV(
    pipeline_rf, param_dist_rf, n_iter=30, cv=5,
    scoring="f1", n_jobs=-1, random_state=42
)
search_rf.fit(X_train, y_train)
pipeline_rf = search_rf.best_estimator_

with mlflow.start_run(run_name="random_forest"):
    y_pred = pipeline_rf.predict(X_test)
    y_proba = pipeline_rf.predict_proba(X_test)[:, 1]

    metrics_rf = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }
    mlflow.log_metrics(metrics_rf)
    mlflow.log_params(search_rf.best_params_)

    mlflow.sklearn.log_model(
        sk_model=pipeline_rf,
        artifact_path="model",
        registered_model_name="ML_CustomerChurn-1821-rf",
        input_example=X_train.iloc[:5],
    )

print(f"RandomForest trained! F1: {metrics_rf['f1']:.2f}, AUC: {metrics_rf['roc_auc']:.2f}")

# ============================================================
# MODELL 2: XGBoost with Hyperparameter-Tuning & Feature Selection
# ============================================================
pipeline_xgb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("feature_selection", SelectFromModel(
        XGBClassifier(random_state=42, eval_metric="logloss"),
        threshold="median"  # behält nur Features über dem Median der Importance
    )),
    ("classifier", XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    )),
])

param_dist_xgb = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__max_depth": [3, 5, 7, 10],
    "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "classifier__subsample": [0.6, 0.8, 1.0],
    "classifier__colsample_bytree": [0.6, 0.8, 1.0],
}

search_xgb = RandomizedSearchCV(
    pipeline_xgb, param_dist_xgb, n_iter=30, cv=5,
    scoring="f1", n_jobs=-1, random_state=42
)
search_xgb.fit(X_train, y_train)
pipeline_xgb = search_xgb.best_estimator_

with mlflow.start_run(run_name="xgboost"):
    y_pred = pipeline_xgb.predict(X_test)
    y_proba = pipeline_xgb.predict_proba(X_test)[:, 1]

    metrics_xgb = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }
    mlflow.log_metrics(metrics_xgb)
    mlflow.log_params(search_xgb.best_params_)

    mlflow.sklearn.log_model(
        sk_model=pipeline_xgb,
        artifact_path="model",
        registered_model_name="ML_CustomerChurn-1821-xgb",
        input_example=X_train.iloc[:5],
    )

print(f"XGBoost trained! F1: {metrics_xgb['f1']:.2f}, AUC: {metrics_xgb['roc_auc']:.2f}")




In [ ]:
from fairlearn.metrics import MetricFrame, selection_rate, demographic_parity_difference, equalized_odds_difference
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, ClassificationPreset

# 7. Fairlearn – Fairness-Check (best model)

sensitive_feature = X_test['cr8b0_gender']  # Spaltenname anpassen
y_pred_xgb = pipeline_xgb.predict(X_test)

metric_frame = MetricFrame(
    metrics={"accuracy": accuracy_score, "selection_rate": lambda yt, yp: yp.mean()},
    y_true=y_test, y_pred=y_pred_xgb, sensitive_features=sensitive_feature,
)
print(metric_frame.by_group)

dpd = demographic_parity_difference(y_test, y_pred_xgb, sensitive_features=sensitive_feature)
eod = equalized_odds_difference(y_test, y_pred_xgb, sensitive_features=sensitive_feature)

with mlflow.start_run(run_name="xgboost_fairness"):
    mlflow.log_metric("demographic_parity_diff", dpd)
    mlflow.log_metric("equalized_odds_diff", eod)

print(f"Demographic Parity Diff: {dpd:.3f}, Equalized Odds Diff: {eod:.3f}")


# 7. Evidently AI – Drift-Report
reference_data = X_train.copy()
reference_data['target'] = y_train.values
reference_data['prediction'] = pipeline_xgb.predict(X_train)

current_data = X_test.copy()
current_data['target'] = y_test.values
current_data['prediction'] = y_pred_xgb

report = Report(metrics=[DataDriftPreset(), ClassificationPreset()])
report.run(reference_data=reference_data, current_data=current_data)
report.save_html("/lakehouse/default/Files/silver/evidently_report.html")

with mlflow.start_run(run_name="xgboost_drift_report"):
    mlflow.log_artifact("/lakehouse/default/Files/silver/evidently_report.html")

# 8.Compare the 2 models

print("\n--- Compare ---")
print(f"RandomForest: F1={metrics_rf['f1']:.3f}, AUC={metrics_rf['roc_auc']:.3f}")
print(f"XGBoost:      F1={metrics_xgb['f1']:.3f}, AUC={metrics_xgb['roc_auc']:.3f}")

In [ ]:
# Load HTML file content
html_content = None
with open("/lakehouse/default/Files/silver/evidently_report.html") as html_file:
    html_content = html_file.read()

displayHTML(html_content)
